In [0]:
from pyspark.sql import functions as F, Window

bronze = spark.table("flightops_dev.bronze.bts_flights")

def hhmm_to_ts(date_col, hhmm_col):
    # BTS times are "HHMM" strings like "745" or "2400"; normalize to a timestamp
    t = F.lpad(F.col(hhmm_col), 4, "0")
    t = F.when(t == "2400", F.lit("2359")).otherwise(t)
    return F.to_timestamp(F.concat_ws(" ", F.col(date_col), t), "yyyy-MM-dd HHmm")

silver = (bronze.select(
        F.to_date("FlightDate").alias("flight_date"),
        F.col("Reporting_Airline").alias("carrier"),
        F.col("Flight_Number_Reporting_Airline").cast("int").alias("flight_number"),
        F.col("Tail_Number").alias("tail_number"),
        F.col("Origin").alias("origin"),
        F.col("OriginCityName").alias("origin_city"),
        F.col("Dest").alias("dest"),
        F.col("DestCityName").alias("dest_city"),
        F.col("CRSDepTime").alias("crs_dep_hhmm"),
        F.col("DepDelay").cast("double").cast("int").alias("dep_delay_min"),
        F.col("ArrDelay").cast("double").cast("int").alias("arr_delay_min"),
        F.col("TaxiOut").cast("double").cast("int").alias("taxi_out_min"),
        F.col("TaxiIn").cast("double").cast("int").alias("taxi_in_min"),
        (F.col("Cancelled").cast("double") == 1).alias("is_cancelled"),
        F.col("CancellationCode").alias("cancellation_code"),
        (F.col("Diverted").cast("double") == 1).alias("is_diverted"),
        F.col("Distance").cast("double").cast("int").alias("distance_miles"),
        F.col("CarrierDelay").cast("double").cast("int").alias("carrier_delay_min"),
        F.col("WeatherDelay").cast("double").cast("int").alias("weather_delay_min"),
        F.col("NASDelay").cast("double").cast("int").alias("nas_delay_min"),
        F.col("SecurityDelay").cast("double").cast("int").alias("security_delay_min"),
        F.col("LateAircraftDelay").cast("double").cast("int").alias("late_aircraft_delay_min"),
        F.col("_source_file"),
        F.col("_ingested_at"))
    .withColumn("sched_dep_local", hhmm_to_ts("flight_date", "crs_dep_hhmm"))
    .withColumn("flight_key", F.sha2(F.concat_ws("|",
        "flight_date", "carrier", "flight_number", "origin", "crs_dep_hhmm"), 256))
)

# Deduplicate: keep the most recently ingested version of each flight
w = Window.partitionBy("flight_key").orderBy(F.col("_ingested_at").desc())
silver = silver.withColumn("_rn", F.row_number().over(w)).filter("_rn = 1").drop("_rn")
print("Silver rows:", silver.count())

In [0]:
checks = {
    "null_keys":          silver.filter("flight_date IS NULL OR carrier IS NULL OR flight_number IS NULL OR origin IS NULL OR dest IS NULL").count(),
    "bad_airport_codes":  silver.filter("length(origin) != 3 OR length(dest) != 3").count(),
    "unparsed_dep_time":  silver.filter("sched_dep_local IS NULL AND NOT is_cancelled").count(),
    "cancelled_with_arr_delay": silver.filter("is_cancelled AND arr_delay_min IS NOT NULL").count(),
}
for name, bad in checks.items():
    print(f"{'PASS' if bad == 0 else 'WARN'}  {name}: {bad}")

assert checks["null_keys"] == 0, "Critical: null business keys in silver"

In [0]:
from delta.tables import DeltaTable

target = "flightops_dev.silver.flights"

if not spark.catalog.tableExists(target):
    silver.write.format("delta").saveAsTable(target)
    spark.sql(f"ALTER TABLE {target} CLUSTER BY (flight_date, carrier)")
else:
    (DeltaTable.forName(spark, target).alias("t")
        .merge(silver.alias("s"), "t.flight_key = s.flight_key")
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute())

print("Silver table rows:", spark.table(target).count())

In [0]:
%sql
SELECT carrier,
       COUNT(*) AS flights,
       ROUND(AVG(arr_delay_min), 1) AS avg_arr_delay,
       ROUND(100 * AVG(CASE WHEN arr_delay_min <= 15 THEN 1 ELSE 0 END), 1) AS on_time_pct,
       ROUND(100 * AVG(CAST(is_cancelled AS INT)), 2) AS cancel_pct
FROM flightops_dev.silver.flights
GROUP BY carrier
ORDER BY flights DESC;